# Fine Tune a pretrained model

## Processing the data

In [ ]:
import torch
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification


checkpoint = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

sequences = [
    "I love to work with pytorch and huggingface transformers",
    "In this section we fine-tune a model with and it's so good to learn fine-tuning"
]

batch = tokenizer(sequences,
                  padding=True,
                  truncation=True,
                  return_tensors="pt")


batch["labels"] = torch.tensor([1, 1])

optimizer = AdamW(model.parameters())

loss = model(**batch).loss
loss.backward()
optimizer.step()

## Loding a dataset from the hub

In [ ]:
import datasets
import huggingface_hub

print(datasets.__version__)
print(huggingface_hub.__version__)

In [ ]:
from datasets import load_dataset

raw_data = load_dataset("glue", "mrpc")
raw_data

In [ ]:
raw_train_data = raw_data["train"]
raw_train_data

In [ ]:
raw_train_data[1]

In [ ]:
raw_train_data.features

In [ ]:
raw_data["train"]["sentence1"][0]

In [ ]:
tokenized_setences_1 = tokenizer(raw_data["train"]["sentence1"][0])
tokenized_setences_2 = tokenizer(raw_train_data["sentence2"][0])

print(f"Tokenize Sentences 1: {tokenized_setences_1}")

In [ ]:
inputs = tokenizer("This is the first sentence.", "This is the second one.")
inputs

In [ ]:
tokenizer.convert_ids_to_tokens(inputs['input_ids'])

In [ ]:
print(type(raw_data["train"]["sentence1"]))
print(type(raw_data["train"]["sentence2"]))

print(raw_data["train"]["sentence1"][:3])
print(raw_data["train"]["sentence2"][:3])


In [ ]:
print(raw_data)

print(raw_data["train"].features)

print(type(raw_data["train"]["sentence1"]))

print(raw_data["train"]["sentence1"][:2])

In [ ]:
tokenized_dataset = tokenizer(
    list(raw_data["train"]["sentence1"]),
    list(raw_data["train"]["sentence2"]),
    padding=True,
    truncation=True,
)

In [ ]:
tokenizer(list(raw_train_data["sentence1"]))

In [ ]:
def tokenize_data(example):
    return tokenizer(list(example["sentence1"]), list(example["sentence2"]), truncation=True)

In [ ]:
list(raw_train_data['sentence2'])

In [ ]:
tokenized_dataset = raw_data.map(tokenize_data, batched=True)

In [ ]:
tokenized_dataset

## Dynamic padding

In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, padding=True)

In [ ]:
samples = tokenized_dataset["train"][:8]
samples

In [ ]:
samples = {k:v for k, v in samples.items() if k not in ["idx", "sentence1", "sentence2"]}
samples

In [ ]:
[len(x) for x in samples["input_ids"]]

In [ ]:
batch = data_collator(samples)
{k: v.shape for k, v in batch.items()}